# Dataset prep

## 🚀 Script execution

In [1]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

%cd /content/

repo_url = "https://github.com/jasonr2048/gh0st-in-the-l00p.git"
repo_dir = "/content/gh0st-in-the-l00p"

if os.path.exists(repo_dir):
    !git -C {repo_dir} fetch
    !git -C {repo_dir} pull
else:
    !git clone {repo_url} {repo_dir}

%cd {repo_dir}

/content
Cloning into '/content/gh0st-in-the-l00p'...
remote: Enumerating objects: 192, done.
remote: Counting objects: 100% (192/192), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 192 (delta 76), reused 160 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (192/192), 168.25 KiB | 3.12 MiB/s, done.
Resolving deltas: 100% (76/76), done.
/content/gh0st-in-the-l00p


In [3]:
# 3. Install deps
!pip install -r tools/requirements.txt -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.5 MB/s eta 0:00:00


In [4]:
# 4. Run on ready images
!python tools/prepare_dataset.py \
    --input_dir  "/content/drive/MyDrive/Gh0st in the Loop/dataset/raw/ready" \
    --output_dir "/content/drive/MyDrive/Gh0st in the Loop/dataset/prepared" \
    --scale 2.5 \
    --resolution 512 \
    --overwrite

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Images found    : 377
Resolution      : 512x512
Scale           : 2.5
Overwrite       : True
Output          : /content/drive/MyDrive/Gh0st in the Loop/dataset/prepared
Rejected        : /content/drive/MyDrive/Gh0st in the Loop/dataset/raw/rejected

Loading YOLOv8 pose model...
Ready.

[1/377] black_crosses/blacal (2).png ... ok
[2/377] black_crosses/blacal (3).png ... ok
[3/377] black_crosses/blacal (4).png ... ok
[4/377] black_crosses/blacal (5).png ... ok
[5/377] black_crosses/blacal (6).png ... ok
[6/377] black_crosses/blacal (7).png ... ok
[7/377] black_crosses/blacal (1).png ... ok
[8/377] red_half_1/DSC08387.JPG ... ok
[9/377] red_half_1/DSC08411.JPG ... ok
[10/377] red_hal

In [6]:
# 4. Run on images processed ofr background removal
!python tools/prepare_dataset.py \
    --input_dir  "/content/drive/MyDrive/Gh0st in the Loop/dataset/bg_removed" \
    --output_dir "/content/drive/MyDrive/Gh0st in the Loop/dataset/prepared" \
    --scale 2.5 \
    --resolution 512 \
    --overwrite

Images found    : 84
Resolution      : 512x512
Scale           : 2.5
Overwrite       : True
Output          : /content/drive/MyDrive/Gh0st in the Loop/dataset/prepared
Rejected        : /content/drive/MyDrive/Gh0st in the Loop/dataset/rejected

Loading YOLOv8 pose model...
Ready.

[1/84] black_mesh/IMG_4038.JPEG ... ok
[2/84] black_mesh/IMG_4032.JPEG ... ok
[3/84] black_mesh/IMG_4040.JPEG ... ok
[4/84] black_mesh/IMG_4042.JPEG ... ok
[5/84] black_mesh/IMG_4039.JPEG ... ok
[6/84] black_mesh/IMG_4028.JPEG ... ok
[7/84] black_mesh/IMG_4036.JPEG ... ok
[8/84] black_mesh/IMG_4045.JPEG ... ok
[9/84] black_mesh/IMG_4044.JPEG ... ok
[10/84] black_mesh/IMG_4048.JPEG ... ok
[11/84] black_mesh/IMG_4041.JPEG ... ok
[12/84] black_mesh/IMG_4046.JPEG ... ok
[13/84] black_mesh/IMG_4033.JPEG ... ok
[14/84] black_mesh/IMG_4025.JPEG ... ok
[15/84] black_mesh/IMG_4043.JPEG ... ok
[16/84] black_mesh/IMG_4030.JPEG ... ok
[17/84] black_mesh/IMG_4027.JPEG ... ok
[18/84] black_mesh/IMG_4031.JPEG ... ok
[19/84]

## 🧪 Debug / Experimentation

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install ultralytics Pillow -q

### Attempt 1: person detection + cropping to face ❌

In [ ]:
"""
prepare_dataset.py

Prepares a dataset of images for StyleGAN2-ADA training by:
  1. Detecting persons using YOLOv8
  2. Cropping to the upper portion of the bounding box (head region)
  3. Expanding to square with padding and resizing to target resolution
  4. Copying undetected/failed images to a 'manual_review' folder

Originals are never modified.

Usage:
    python tools/prepare_dataset.py \
        --input_dir /path/to/raw/images \
        --output_dir /path/to/output \
        --resolution 512 \
        --padding 0.3 \
        --head_fraction 0.5 \
        --sample 20

Requirements:
    pip install ultralytics Pillow
    (ultralytics includes opencv)
"""

import argparse
import random
import shutil
import sys
from pathlib import Path

from PIL import Image
from ultralytics import YOLO


# ── Config ────────────────────────────────────────────────────────────────────

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}

# ── Helpers ───────────────────────────────────────────────────────────────────

def detect_person(image_path: Path, model):
    """
    Run YOLOv8 person detection. Returns the largest person bounding box
    as (x1, y1, x2, y2) in pixel coords, or None if no person found.
    """
    results = model(str(image_path), classes=[0], verbose=False)  # class 0 = person
    boxes = results[0].boxes

    if boxes is None or len(boxes) == 0:
        return None

    # Pick largest bounding box by area
    best = max(boxes.xyxy.tolist(), key=lambda b: (b[2] - b[0]) * (b[3] - b[1]))
    return tuple(map(int, best))


def make_head_crop(image: Image.Image, x1: int, y1: int, x2: int, y2: int,
                   padding: float, head_fraction: float):
    """
    Crop to the upper head_fraction of the person bounding box,
    expand by padding, make square, clamp to image bounds.
    """
    w, h = image.size

    # Take upper portion of person box as head region
    box_h = y2 - y1
    head_y2 = y1 + int(box_h * head_fraction)

    # Centre of head region
    cx = (x1 + x2) // 2
    cy = (y1 + head_y2) // 2

    # Square side from head region dimensions + padding
    head_w = x2 - x1
    head_h = head_y2 - y1
    side = int(max(head_w, head_h) * (1 + padding))
    half = side // 2

    # Clamp to image bounds
    left   = max(0, cx - half)
    top    = max(0, cy - half)
    right  = min(w, cx + half)
    bottom = min(h, cy + half)

    cropped = image.crop((left, top, right, bottom))

    if cropped.width == 0 or cropped.height == 0:
        return None

    # Make strictly square by cropping the longer side from centre
    cw, ch = cropped.size
    if cw != ch:
        min_side = min(cw, ch)
        cropped = cropped.crop((0, 0, min_side, min_side))

    return cropped


def process_image(path: Path, output_dir: Path, review_dir: Path,
                  model, resolution: int, padding: float, head_fraction: float) -> str:
    """
    Process a single image. Returns 'ok', 'review', or 'error'.
    """
    try:
        image = Image.open(path).convert("RGB")
        box = detect_person(path, model)

        if box is None:
            shutil.copy2(path, review_dir / path.name)
            return "review"

        x1, y1, x2, y2 = box
        cropped = make_head_crop(image, x1, y1, x2, y2, padding, head_fraction)

        if cropped is None:
            shutil.copy2(path, review_dir / path.name)
            return "review"

        resized = cropped.resize((resolution, resolution), Image.LANCZOS)
        out_path = output_dir / (path.stem + ".png")
        resized.save(out_path)
        return "ok"

    except Exception as e:
        print(f"\n  ERROR processing {path.name}: {e}")
        return "error"


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Prepare face dataset for StyleGAN training.")
    parser.add_argument("--input_dir",     required=True,            help="Folder containing raw images (read-only)")
    parser.add_argument("--output_dir",    required=True,            help="Folder for processed images")
    parser.add_argument("--resolution",    type=int,   default=512,  help="Output resolution (default: 512)")
    parser.add_argument("--padding",       type=float, default=0.3,  help="Padding around head region (default: 0.3)")
    parser.add_argument("--head_fraction", type=float, default=0.5,  help="Fraction of person box treated as head (default: 0.5)")
    parser.add_argument("--sample",        type=int,   default=None, help="Process only a random sample of N images")
    args = parser.parse_args()

    input_dir  = Path(args.input_dir)
    output_dir = Path(args.output_dir)
    review_dir = output_dir / "manual_review"

    if not input_dir.exists():
        print(f"Input directory not found: {input_dir}")
        sys.exit(1)

    output_dir.mkdir(parents=True, exist_ok=True)
    review_dir.mkdir(parents=True, exist_ok=True)

    # Collect images
    images = [p for p in input_dir.rglob("*") if p.suffix.lower() in SUPPORTED_EXTENSIONS]
    if not images:
        print("No supported images found.")
        sys.exit(0)

    if args.sample is not None:
        images = random.sample(images, min(args.sample, len(images)))

    print(f"Images to process : {len(images)}")
    print(f"Resolution        : {args.resolution}x{args.resolution}")
    print(f"Padding           : {args.padding}")
    print(f"Head fraction     : {args.head_fraction}")
    print(f"Output            : {output_dir}")
    print(f"Manual review     : {review_dir}")
    print()

    # Load YOLOv8n — smallest/fastest, downloads automatically on first run
    print("Loading YOLOv8 model...")
    model = YOLO("yolov8n.pt")
    print("Ready.\n")

    counts = {"ok": 0, "review": 0, "error": 0}

    for i, path in enumerate(images, 1):
        print(f"[{i}/{len(images)}] {path.name} ... ", end="", flush=True)
        result = process_image(path, output_dir, review_dir, model,
                               args.resolution, args.padding, args.head_fraction)
        counts[result] += 1
        print(result)

    print()
    print("── Summary ──────────────────────────────")
    print(f"  Auto-cropped  : {counts['ok']}")
    print(f"  Manual review : {counts['review']}  →  {review_dir}")
    print(f"  Errors        : {counts['error']}")
    print(f"  Total         : {len(images)}")

    if counts["review"] > 0:
        print()
        print("  Tip: try adjusting --head_fraction or --padding and re-running.")


if __name__ == "__main__":
    import sys
    sys.argv = [
        "prepare_dataset.py",
        "--input_dir", "/content/drive/MyDrive/Gh0st in the Loop/dataset/raw",
        "--output_dir", "/content/drive/MyDrive/Gh0st in the Loop/dataset/prepared",
        "--sample", "10",
        "--padding", "0.3",
        "--head_fraction", "0.5"
    ]
    main()

In [ ]:
from ultralytics import YOLO
from PIL import Image, ImageDraw
import random
from pathlib import Path

def debug_detections(input_dir, n=5, head_fraction=0.5):
    model = YOLO("yolov8n.pt")
    images = [p for p in Path(input_dir).rglob("*")
              if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}]
    sample = random.sample(images, min(n, len(images)))

    for path in sample:
        image = Image.open(path).convert("RGB")
        results = model(str(path), classes=[0], verbose=False)
        boxes = results[0].boxes

        draw = ImageDraw.Draw(image)

        if boxes and len(boxes) > 0:
            # Draw all detected person boxes in blue
            for box in boxes.xyxy.tolist():
                x1, y1, x2, y2 = map(int, box)
                draw.rectangle([x1, y1, x2, y2], outline="blue", width=4)
                # Draw head fraction region in red
                head_y2 = y1 + int((y2 - y1) * head_fraction)
                draw.rectangle([x1, y1, x2, head_y2], outline="red", width=4)
        else:
            draw.text((10, 10), "NO DETECTION", fill="red")

        # Show inline in Colab
        display(image.resize((400, int(400 * image.height / image.width))))

debug_detections(
    "/content/drive/MyDrive/Gh0st in the Loop/dataset/raw",
    n=15,
    head_fraction=0.5
)

### Attempt 2: face detection via pose and YOLO

In [ ]:
from ultralytics import YOLO
from PIL import Image, ImageDraw
import random
from pathlib import Path

def debug_pose(input_dir, n=5):
    model = YOLO("yolov8n-pose.pt")  # pose model, downloads automatically
    images = [p for p in Path(input_dir).rglob("*")
              if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}]
    sample = random.sample(images, min(n, len(images)))

    for path in sample:
        image = Image.open(path).convert("RGB")
        results = model(str(path), verbose=False)

        draw = ImageDraw.Draw(image)

        if results[0].keypoints is not None:
            kpts = results[0].keypoints.xy[0].tolist()  # first person
            # Keypoints 0-4: nose, left eye, right eye, left ear, right ear
            head_kpts = [(int(x), int(y)) for x, y in kpts[:5] if x > 0 and y > 0]

            for x, y in head_kpts:
                draw.ellipse([x-10, y-10, x+10, y+10], fill="red")

            if head_kpts:
                xs = [p[0] for p in head_kpts]
                ys = [p[1] for p in head_kpts]
                # Draw bounding box around head keypoints
                draw.rectangle([min(xs)-40, min(ys)-40, max(xs)+40, max(ys)+40],
                              outline="red", width=4)
        else:
            draw.text((10, 10), "NO KEYPOINTS", fill="red")

        display(image.resize((400, int(400 * image.height / image.width))))

debug_pose(
    "/content/drive/MyDrive/Gh0st in the Loop/dataset/raw",
    n=6
)

### Attempt 3: combine face keypoints & person bounding-box

In [ ]:
from ultralytics import YOLO
from PIL import Image, ImageDraw, ImageFile
import random
from pathlib import Path
import numpy as np

# Allow very large images and warn instead of error
Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True

TARGET_LONG_EDGE = 1024  # downsample large images to this before processing


def downsample_if_needed(image):
    """Downsample to TARGET_LONG_EDGE on the long side, preserving aspect ratio."""
    w, h = image.size
    long_edge = max(w, h)
    if long_edge > TARGET_LONG_EDGE:
        scale = TARGET_LONG_EDGE / long_edge
        image = image.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return image


def sample_background_colour(image, crop_left, crop_top, crop_right, crop_bottom, hits):
    """Sample background from top corners of crop, or bottom if top is out of bounds."""
    w, h = image.size
    img_arr = np.array(image)
    samples = []

    if "top" not in hits:
        ty = max(0, crop_top)
        samples.append(img_arr[ty, max(0, min(w-1, crop_left))].tolist())
        samples.append(img_arr[ty, max(0, min(w-1, crop_right))].tolist())
    else:
        by = min(h-1, crop_bottom)
        samples.append(img_arr[by, max(0, min(w-1, crop_left))].tolist())
        samples.append(img_arr[by, max(0, min(w-1, crop_right))].tolist())

    return tuple(int(x) for x in np.mean(samples, axis=0)[:3])


def pad_to_square(image, crop_left, crop_top, crop_right, crop_bottom, bg_colour):
    w, h = image.size
    clamped = image.crop((
        max(0, crop_left), max(0, crop_top),
        min(w, crop_right), min(h, crop_bottom)
    ))
    target_size = max(crop_right - crop_left, crop_bottom - crop_top)
    square = Image.new("RGB", (target_size, target_size), bg_colour)

    # Paste at the position matching where content actually was in the crop box
    paste_x = max(0, -crop_left)  # if crop_left < 0, content starts inset
    paste_y = max(0, -crop_top)
    square.paste(clamped, (paste_x, paste_y))
    return square


def debug_combined_crop(input_dir, n=5, scale=3.0):
    pose_model = YOLO("yolov8n-pose.pt")

    images = [p for p in Path(input_dir).rglob("*")
              if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}]
    sample = random.sample(images, min(n, len(images)))

    edge_hits = []

    for i, path in enumerate(sample, 1):
        print(f"\n{'='*50}")
        print(f"[{i}/{len(sample)}] {path.name}")
        print(f"{'='*50}")

        image = Image.open(path).convert("RGB")
        orig_w, orig_h = image.size
        image = downsample_if_needed(image)
        w, h = image.size

        # Scale factor between original and downsampled
        scale_x = w / orig_w
        scale_y = h / orig_h

        # Detection on clean image
        pose_results = pose_model(str(path), verbose=False)
        if pose_results[0].keypoints is None:
            print("❌ No keypoints detected → skipping")
            continue

        kpts = pose_results[0].keypoints.xy[0].tolist()
        head_kpts = [(int(x * scale_x), int(y * scale_y)) for x, y in kpts[:5] if x > 0 and y > 0]
        if not head_kpts:
            print("❌ No head keypoints → skipping")
            continue

        kpt_xs = [p[0] for p in head_kpts]
        kpt_ys = [p[1] for p in head_kpts]
        cx = (min(kpt_xs) + max(kpt_xs)) // 2
        cy = (min(kpt_ys) + max(kpt_ys)) // 2

        span = max(max(kpt_xs) - min(kpt_xs), max(kpt_ys) - min(kpt_ys))
        half = int(span * scale / 2)

        crop_left   = cx - half
        crop_right  = cx + half
        crop_top    = cy - half
        crop_bottom = cy + half

        hits = []
        if crop_left < 0:   hits.append("left")
        if crop_right > w:  hits.append("right")
        if crop_top < 0:    hits.append("top")
        if crop_bottom > h: hits.append("bottom")

        # --- Draw overlay on a COPY ---
        display_image = image.copy()
        draw = ImageDraw.Draw(display_image)
        box_colour = "red" if hits else "green"
        draw.rectangle([
            max(0, crop_left), max(0, crop_top),
            min(w, crop_right), min(h, crop_bottom)
        ], outline=box_colour, width=4)
        for x, y in head_kpts:
            draw.ellipse([x-8, y-8, x+8, y+8], fill="yellow")
        draw.ellipse([cx-8, cy-8, cx+8, cy+8], fill="white")

        print("📷 Original with crop overlay:")
        if hits:
            print(f"  ⚠️  Edge hit: {', '.join(hits)}")
            edge_hits.append((path.name, hits))
        else:
            print("  ✅ Clean crop, no padding needed")
        display(display_image.resize((400, int(400 * h / w))))

        # --- Padded crop result from clean image ---
        bg = sample_background_colour(image, crop_left, crop_top, crop_right, crop_bottom, hits)
        result = pad_to_square(image, crop_left, crop_top, crop_right, crop_bottom, bg)
        print(f"✂️  Resulting crop (bg colour sampled: {bg}):")
        display(result.resize((400, 400)))

    print(f"\n{'='*50}")
    if edge_hits:
        print("⚠️  EDGE HITS — padding was applied for:")
        for name, sides in edge_hits:
            print(f"  {name}: {', '.join(sides)}")
    else:
        print("✅ No edge hits in this sample.")


debug_combined_crop(
    "/content/drive/MyDrive/Gh0st in the Loop/dataset/raw",
    n=10,
    scale=2.5
)

In [ ]:
import os

base = "/content/drive/MyDrive/Gh0st in the Loop/dataset/raw"
for i in range(24):
    folder = os.path.join(base, chr(65 + i))
    os.makedirs(folder, exist_ok=True)

print("Done:", os.listdir(base))

In [ ]:
import os
import shutil

for i in range(24):
    folder = chr(65 + i)
    if os.path.exists(folder):
        shutil.rmtree(folder)

print("Done")

### Debugging failures

In [ ]:
FAILURES = [
    "AB/IMG_20260310_224124_764.jpg",  # Processed fine
    "B_black_feathers/DSC06578.JPG",  # Errored
    "O_crochet_flower_back/IMG_3834.JPEG",
    "O_crochet_flower_back/IMG_3828.JPEG",
    "Z/IMG_20260310_230811_599.jpg",
    "Z/IMG_20260310_230813_810.jpg",
    "W/IMG_20260310_233004_072.jpg",
    "W/IMG_20260310_232958_263.jpg",
    "W/IMG_20260310_232951_988.jpg",
    "W/IMG_20260310_233006_760.jpg",
    "W/IMG_20260310_233002_557.jpg",
    "W/IMG_20260310_233005_441.jpg",
    "Y/IMG_20260310_231604_269.jpg",
    "Y/IMG_20260310_231537_673.jpg",
    "X/IMG_20260310_232207_150.jpg",
    "X/IMG_20260310_232158_872.jpg",
    "AB/IMG_20260310_224032_931.jpg",
    "AB/IMG_20260310_224216_018.jpg",
]

In [ ]:
from PIL import Image, ImageOps
from ultralytics import YOLO
from pathlib import Path
import numpy as np

pose_model = YOLO("yolov8n-pose.pt")

DETECTION_LONG_EDGE = 1024

def downsample_for_detection(image):
    w, h = image.size
    long_edge = max(w, h)
    if long_edge <= DETECTION_LONG_EDGE:
        return image, 1.0, 1.0
    scale = DETECTION_LONG_EDGE / long_edge
    small = image.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return small, w / small.width, h / small.height

input_dir = Path("/content/drive/MyDrive/Gh0st in the Loop/Dataset/raw")

for rel in FAILURES:
    path = input_dir / rel
    print(f"\n{'='*50}")
    print(f"{rel}")

    # What the script does
    image_raw = Image.open(path).convert("RGB")
    small, scale_x, scale_y = downsample_for_detection(image_raw)
    tmp_path = Path("/tmp") / path.name
    small.save(tmp_path)
    results_script = pose_model(str(tmp_path), verbose=False)
    tmp_path.unlink(missing_ok=True)
    script_detections = results_script[0].keypoints.xy.shape[0]

    # What the debug cell does (direct path, no temp file)
    results_direct = pose_model(str(path), verbose=False)
    direct_detections = results_direct[0].keypoints.xy.shape[0]

    print(f"  script path (temp file): {script_detections} detections")
    print(f"  direct path (original) : {direct_detections} detections")

    # Show what the temp file looked like
    print("  Temp file (what YOLO saw in script):")
    display(small.resize((400, int(400 * small.height / small.width))))

    # Show correctly oriented original
    print("  Original with EXIF correction:")
    image_corrected = ImageOps.exif_transpose(image_raw)
    display(image_corrected.resize((400, int(400 * image_corrected.height / image_corrected.width))))